In [8]:
import torch
import pandas as pd

In [165]:
df = pd.read_csv("data/Dataset_English_Hindi.csv").dropna().drop_duplicates()
df.head()

,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [166]:
# remove puncutations
import re

df['English'] = df['English'].apply(lambda text: re.sub(r'[^\w\s]', '', text)).str.lower()
df['Hindi'] = df['Hindi'].apply(lambda text: re.sub(r'[^\w\s]', '', text)).str.lower()

In [167]:
df["Hindi"] = df["Hindi"].apply(lambda text: f"<sos> {text} <eos>")

In [168]:
df_cpy = df.iloc[:25000]
source_tokens, target_tokens = ['<unk>'], ['<unk>']

for source, target in df_cpy.values:
    for text in source.split(" "):
        if text not in source_tokens:
            source_tokens.append(text)
    
    for text in target.split(" "):
        if text not in target_tokens:
            target_tokens.append(text)

In [171]:
# encoding the sentences and forming the dataset

source_encodings, target_encodings = [], []

for source, target in df_cpy.values:
    source_encoding, target_encoding = [], []

    for text in source.split():
        source_encoding.append(source_tokens.index(text))
    source_encodings.append(source_encoding)

    for text in target.split():
        target_encoding.append(target_tokens.index(text))
    target_encodings.append(target_encoding)

In [172]:
import torch.nn.functional as F

padded_source_encodings, padded_target_encodings = [], []

for i in source_encodings:
    padded_source_encodings.append(
        F.pad(
            torch.tensor(i), pad=(max([len(i) for i in source_encodings]) - len(i), 0), mode='constant', value=0
        ))

    padded_target_encodings.append(
        F.pad(torch.tensor(i), pad=(max([len(i) for i in target_encodings]) - len(i), 0), mode='constant', value=0)
    )

In [173]:
class EncoderDecoder(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()

        # Embeddings
        self.encoder_embeddings = torch.nn.Embedding(
            # NOTE —→ sample values
            num_embeddings=len(source_tokens),
            embedding_dim=512
        )
        self.decoder_embeddings = torch.nn.Embedding(
            # NOTE —→ sample values
            num_embeddings=len(target_tokens),
            embedding_dim=512
        )

        # Sequential Models
        self.encoder_rnn = torch.nn.RNN(
            # NOTE —→ sample values
            batch_first=True, input_size=512, hidden_size=250
        )
        self.decoder_rnn = torch.nn.RNN(
            # NOTE —→ sample values
            batch_first=True, input_size=512, hidden_size=250
        )

        # Linear Layer
        self.output = torch.nn.Linear(in_features=250, out_features=len(target_tokens))
    
    def forward(self, x: torch.Tensor, y: torch.Tensor):
        # Encoder Block
        source_embeddings = self.encoder_embeddings(x).unsqueeze(dim=0)
        _, final_hidden_state = self.encoder_rnn(source_embeddings) # NOTE: sample value

        # Decoder Block
        target_embeddings = self.decoder_embeddings(y).unsqueeze(dim=0)
        all_hidden_states, _ = self.decoder_rnn(target_embeddings, final_hidden_state) # torch.Size([1, 228, 10])

        logits = self.output(all_hidden_states)
        
        return logits
        
model = EncoderDecoder()

In [174]:
model(padded_source_encodings[0], padded_target_encodings[0]).shape

torch.Size([1, 315, 23776])

In [176]:
padded_target_encodings[0].shape

torch.Size([315])

In [139]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [180]:
for i in range(5):
    epoch_loss = 0
    for j in range(len(padded_source_encodings)):
        optimizer.zero_grad()

        # forward_pass
        output = model(padded_source_encodings[j], padded_target_encodings[j])

        # loss
        loss = criterion(output[0], padded_target_encodings[j])
        loss.backward()

        optimizer.step()
        epoch_loss += loss.item()
    print(epoch_loss / len(padded_source_encodings))

KeyboardInterrupt: 